# 01 — Exploratory Data Analysis

**Boréal Marché — Customer Intelligence Platform**

This notebook explores the UCI Online Retail II dataset (re-skinned as Boréal Marché)
before any modelling. All business logic lives in `src/` — this notebook only calls
those modules and visualises their output.

## Contents
1. Data loading and cleaning summary
2. Transaction volume over time
3. Customer distribution (RFM)
4. Feature distributions and correlations
5. Class balance (will_return)
6. Geographic breakdown

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingest import load_config
from src.features import load_transactions, create_labels, build_features, FEATURE_NAMES

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Data Loading and Cleaning Summary

In [ ]:
cfg = load_config()
df = load_transactions(cfg)
print(f"Transactions: {len(df):,} rows × {len(df.columns)} columns")
print(f"Date range: {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['Customer ID'].nunique():,}")
print(f"Unique products: {df['StockCode'].nunique():,}")
df.head()

## 2. Transaction Volume Over Time

In [ ]:
daily = df.set_index("InvoiceDate").resample("W")["Invoice"].nunique()
fig, ax = plt.subplots()
daily.plot(ax=ax, title="Weekly Invoice Count")
ax.axvline(pd.Timestamp(cfg["labelling"]["cutoff_date"]), color="red", ls="--", label="Cutoff")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Feature Distributions

In [ ]:
cutoff = cfg["labelling"]["cutoff_date"]
window = cfg["labelling"]["label_window_days"]
obs_df, label_df, y = create_labels(df, cutoff, window)
X = build_features(obs_df, df, cutoff)

print(f"Feature matrix: {X.shape[0]} customers × {X.shape[1]} features")
print(f"Class balance: {y.value_counts().to_dict()}")
X.describe().T

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, col in zip(axes.flat, ["recency_days", "frequency", "monetary_total",
                                "avg_days_between_orders", "avg_basket_value",
                                "category_entropy", "cancellation_rate",
                                "weekend_purchase_ratio", "n_countries"]):
    X[col].hist(bins=40, ax=ax, edgecolor="white")
    ax.set_title(col)
plt.suptitle("Feature Distributions", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 5. Class Balance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
y.value_counts().plot.bar(ax=ax, color=["#e74c3c", "#2ecc71"])
ax.set_xticklabels(["Churned (0)", "Returned (1)"], rotation=0)
ax.set_title("Class Balance: will_return")
for i, v in enumerate(y.value_counts()):
    ax.text(i, v + 20, f"{v} ({v/len(y):.1%})", ha="center")
plt.tight_layout()
plt.show()

## 6. Geographic Breakdown

In [ ]:
country_counts = df.groupby("Country")["Customer ID"].nunique().sort_values(ascending=False)
print(f"Domestic (UK): {country_counts.get('United Kingdom', 0)} customers")
print(f"International: {country_counts.drop('United Kingdom', errors='ignore').sum()} customers")
print(f"\nTop 10 countries:")
country_counts.head(10)